In [3]:
import pandas as pd

df = pd.read_csv("/cdvae_final/to_update_unzipped/mp20_test.csv")

print(f"Total rows     : {len(df)}")
print(f"Columns        : {df.columns.tolist()}")
print(f"\nFirst row:")
print(df.iloc[0])

print(f"\nEnergy stats:")
print(df[["formation_energy_per_atom", "e_above_hull", "band_gap"]].describe())

print(f"\nMissing values:")
print(df[["formation_energy_per_atom", "e_above_hull", "band_gap"]].isna().sum())

Total rows     : 9046
Columns        : ['Unnamed: 0', 'material_id', 'formation_energy_per_atom', 'band_gap', 'pretty_formula', 'e_above_hull', 'elements', 'cif', 'spacegroup.number']

First row:
Unnamed: 0                                                                6000
material_id                                                           mp-10009
formation_energy_per_atom                                            -0.575092
band_gap                                                                 0.898
pretty_formula                                                            GaTe
e_above_hull                                                               0.0
elements                                                          ['Ga', 'Te']
cif                          # generated using pymatgen\ndata_GaTe\n_symmet...
spacegroup.number                                                          194
Name: 0, dtype: object

Energy stats:
       formation_energy_per_atom  e_above_hull     band

In [9]:
# Cell 1 — imports
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch_geometric.data import Data
from pymatgen.core import Structure

# Cell 2 — your paths (adjust these)
CSV_PATH     = "/cdvae_final/to_update_unzipped/mp20_train.csv"   
OUT_DIR      = "/cdvae_final/to_update_unzipped/dataset_files"
BATCH_SIZE   = 100
RADIUS       = 6.0
MAX_NEIGHBORS= 12

os.makedirs(OUT_DIR, exist_ok=True)

# Cell 3 — load and check CSV
df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df[["formation_energy_per_atom","e_above_hull","band_gap"]].describe())

Rows: 27136
Columns: ['Unnamed: 0', 'material_id', 'formation_energy_per_atom', 'band_gap', 'pretty_formula', 'e_above_hull', 'elements', 'cif', 'spacegroup.number']
       formation_energy_per_atom  e_above_hull      band_gap
count               27136.000000  27136.000000  27136.000000
mean                   -1.219803      0.017478      0.791848
std                     1.029393      0.023265      1.417775
min                    -5.153569      0.000000      0.000000
25%                    -2.013250      0.000000      0.000000
50%                    -0.814771      0.004073      0.000000
75%                    -0.400797      0.031026      1.142100
max                     0.079825      0.079999     17.902300


In [11]:
# Cell 4 — graph builder function
def structure_to_graph(structure, radius=6.0, max_neighbors=12):
    N       = len(structure)
    species = [site.specie.Z for site in structure]
    frac    = np.array([site.frac_coords % 1.0 
                        for site in structure], dtype=np.float32)
    lattice = np.array(structure.lattice.matrix, dtype=np.float32)

    all_neighbors = structure.get_all_neighbors(r=radius, include_index=True)
    src_list, dst_list, dist_list = [], [], []

    for i, neighbors in enumerate(all_neighbors):
        neighbors = sorted(neighbors, key=lambda x: x[1])[:max_neighbors]
        for nb in neighbors:
            src_list.append(i)
            dst_list.append(nb[2])
            dist_list.append(nb[1])

    # Fallback for isolated atoms
    if len(src_list) == 0:
        for i in range(N):
            for j in range(N):
                if i != j:
                    src_list.append(i)
                    dst_list.append(j)
                    dist_list.append(1.0)

    return Data(
        x          = torch.tensor(species,   dtype=torch.long),
        pos        = torch.tensor(frac,      dtype=torch.float32),
        lattice    = torch.tensor(lattice,   dtype=torch.float32),
        edge_index = torch.tensor([src_list, dst_list], dtype=torch.long),
        edge_attr  = torch.tensor(dist_list, dtype=torch.float32),
    )

In [13]:
# Cell 5 — build all graphs
batch    = []
batch_id = 0
n_ok     = 0
n_fail   = 0
skipped  = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building graphs"):
    try:
        structure = Structure.from_str(row["cif"], fmt="cif")
        graph     = structure_to_graph(structure, RADIUS, MAX_NEIGHBORS)

        graph.y = torch.tensor(
            [row["formation_energy_per_atom"]
             if pd.notna(row["formation_energy_per_atom"])
             else float("nan")], dtype=torch.float32)

        graph.e_above_hull = torch.tensor(
            [row["e_above_hull"]
             if pd.notna(row["e_above_hull"])
             else float("nan")], dtype=torch.float32)

        graph.band_gap = torch.tensor(
            [row["band_gap"]
             if pd.notna(row["band_gap"])
             else float("nan")], dtype=torch.float32)

        batch.append(graph)
        n_ok += 1

        if len(batch) == BATCH_SIZE:
            torch.save(batch, os.path.join(OUT_DIR, f"batch_{batch_id:04d}.pt"))
            batch_id += 1
            batch = []

    except Exception as e:
        n_fail += 1
        skipped.append(idx)

# Save last partial batch
if batch:
    torch.save(batch, os.path.join(OUT_DIR, f"batch_{batch_id:04d}.pt"))
    batch_id += 1

print(f"\nDone!")
print(f"  Saved  : {n_ok} graphs in {batch_id} files")
print(f"  Failed : {n_fail}")
if skipped:
    print(f"  Skipped indices: {skipped[:20]}")

Building graphs:   0%|                                                               | 5/27136 [00:00<50:23,  8.97it/s]C:\Users\shekh\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
Building graphs:   0%|                                                              | 12/27136 [00:00<23:39, 19.11it/s]C:\Users\shekh\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
C:\Users\shekh\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision


Done!
  Saved  : 27136 graphs in 272 files
  Failed : 0


In [15]:
# Cell 6 — verify
batch = torch.load(os.path.join(OUT_DIR, "batch_0000.pt"), weights_only=False)
g     = batch[0]
print(f"Graphs in batch  : {len(batch)}")
print(f"Atoms            : {g.x.shape} → {g.x[:5].tolist()}")
print(f"Frac coords      : {g.pos.shape} | range [{g.pos.min():.3f}, {g.pos.max():.3f}]")
print(f"Edges            : {g.edge_index.shape}")
print(f"Edge dist range  : [{g.edge_attr.min():.2f}, {g.edge_attr.max():.2f}] Å")
print(f"Formation energy : {g.y.item():.4f}        {'✅' if not torch.isnan(g.y).any() else '❌'}")
print(f"E above hull     : {g.e_above_hull.item():.4f}  {'✅' if not torch.isnan(g.e_above_hull).any() else '❌'}")
print(f"Band gap         : {g.band_gap.item():.4f}  {'✅' if not torch.isnan(g.band_gap).any() else '❌'}")

Graphs in batch  : 100
Atoms            : torch.Size([12]) → [11, 11, 11, 25, 27]
Frac coords      : torch.Size([12, 3]) | range [0.002, 1.000]
Edges            : torch.Size([2, 144])
Edge dist range  : [1.87, 3.13] Å
Formation energy : -1.6375        ✅
E above hull     : 0.0430  ✅
Band gap         : 0.2133  ✅


In [20]:
bad_graphs = []
pt_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith(".pt")])

for i, fname in enumerate(pt_files):
    batch = torch.load(os.path.join(OUT_DIR, fname), weights_only=False)
    for j, g in enumerate(batch):
        global_idx = i * 100 + j
        if (g.x.shape[0] == 0 or
            g.edge_index.shape[1] == 0 or
            torch.isnan(g.pos).any() or
            torch.isnan(g.lattice).any()):
            bad_graphs.append(global_idx)

print(f"Bad graphs: {len(bad_graphs)}")
print(set(bad_graphs))

Bad graphs: 0
set()
